# Batch face swap (InSwapper, hair/clothes/background preserved)

Local ArcFace + InSwapper face swap. **Deterministic, no diffusion, no prompt-following, no masking guesswork** -- it detects the face, extracts identity from the donor photo, swaps just the face region, and blends it back with a feathered ellipse mask. Hair, clothing and background are never touched because nothing outside that ellipse is ever regenerated.

Upload 13 body/face pairs and run all of them in one pass.

**3 cells: Setup -> Upload -> Run.**

## 1 · Setup

In [ ]:
from pathlib import Path
import subprocess, os

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-only"

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU, then Run all."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True
)

os.environ["HEADSWAP_REPO"] = str(REPO)
os.environ["INSWAP_REPO_BRANCH"] = BRANCH
# Drive-cached models -- survives session restarts, no re-downloading every run.
os.environ["INSWAP_CACHE"] = "/content/drive/MyDrive/headswap_inswap"
os.environ["INSWAP_LOCAL_CACHE"] = "/content/inswap_cache"

from google.colab import drive
drive.mount("/content/drive")

# Reuses the existing, already-tested setup script -- installs insightface,
# onnxruntime-gpu, and downloads the InSwapper-128 + buffalo_l models.
subprocess.run(["bash", str(REPO / "scripts" / "setup_inswapper_colab.sh")], check=True)
print("Setup complete.")


## 2 · Upload 13 pairs

Upload all 13 **body/scene** photos first (select all at once), then all 13 **face/identity donor** photos in the *same order*. The cell prints the resulting pairing so you can check it before running anything.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image
import io

UPLOAD_DIR = Path("/content/face_swap_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Upload all BODY/scene photos (select all 13 at once)")
body_uploads = files.upload()
print()
print("Upload all FACE/identity donor photos, in the SAME ORDER as the bodies above (select all 13 at once)")
face_uploads = files.upload()

body_names = sorted(body_uploads.keys())
face_names = sorted(face_uploads.keys())
assert len(body_names) == len(face_names), (
    f"Got {len(body_names)} body images and {len(face_names)} face images -- "
    "these must match 1:1. Re-run this cell with matching sets."
)

pairs = []
print()
print("Pairing (verify this matches what you intended):")
for i, (bn, fn) in enumerate(zip(body_names, face_names), start=1):
    bp = UPLOAD_DIR / f"body_{i:02d}.png"
    fp = UPLOAD_DIR / f"face_{i:02d}.png"
    Image.open(io.BytesIO(body_uploads[bn])).convert("RGB").save(bp)
    Image.open(io.BytesIO(face_uploads[fn])).convert("RGB").save(fp)
    pairs.append((bp, fp))
    print(f"  {i:2d}. body={bn!r}  <->  face={fn!r}")

print(f"\n{len(pairs)} pairs ready.")


## 3 · Run all pairs

InSwapper only (`krea2_refine=False`) -- keeps this deterministic and simple, no diffusion prompt/masking behaviour to fight. Loads the model once, then runs every pair.

In [ ]:
from pathlib import Path
import sys, os, time

REPO = Path("/content/headswap_V2")
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from headswap.inswap.pipeline import InSwapPipeline
from PIL import Image
from IPython.display import display, Markdown

CACHE_DIR = Path(os.environ.get("INSWAP_LOCAL_CACHE", "/content/inswap_cache"))
OUT_DIR = Path("/content/face_swap_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipe = InSwapPipeline(
    cache_dir=CACHE_DIR,
    engine="inswapper",
    restorer="none",
    device="cuda",
    blend_strength=1.0,
    color_match_strength=0.25,
    krea2_refine=False,  # simple, deterministic, no diffusion masking/prompt issues
)
pipe.load()

results = []
t0 = time.perf_counter()
for i, (body_path, face_path) in enumerate(pairs, start=1):
    body = Image.open(body_path).convert("RGB")
    face = Image.open(face_path).convert("RGB")
    pair_out = OUT_DIR / f"pair_{i:02d}"
    res = pipe.run(body, face, out_dir=pair_out, save_intermediates=False)
    results.append(res)
    result_path = pair_out / "result.png"
    print(f"[{i}/{len(pairs)}] {res.latency_s:.1f}s -> {result_path}")

print(f"\nAll {len(pairs)} pairs done in {time.perf_counter() - t0:.1f}s total.")

for i, res in enumerate(results, start=1):
    display(Markdown(f"### Pair {i}"))
    display(res.image)
